# Dataset Exploration — Disponibilità dataset e tipologie di dato

Parte 1 di 3 (vedi anche `lesion_quality.ipynb` e `clinical_metadata_raw.ipynb`).

## Dataset e tipologie di dato

Esplorazione diretta di `data/clinical_connectome/derivatives`: quanti dataset, quali tipologie di dato per ciascuno, quanti file per soggetto, e verifica di consistenza (file attesi, duplicati, copertura incrociata).

In [1]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from collections import defaultdict

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Path dinamici relativi alla posizione del progetto
project_root = '/Users/emmatosato/Local Projects/Local PhD Projects/NEMESIS_fs'
DERIV_ROOT = Path(project_root) / "data" / "clinical_connectome" / "derivatives"


**Dataset** = cartella a profondità 2 sotto `derivatives` (site/cohort, es. `UNIPD/WashU`).

**Tipologia di dato** = sua sottocartella di primo livello (`manual_masks`, `sdc`, `features`).

In [2]:
def discover_datasets(deriv_root: Path) -> dict[str, Path]:
    return {
        f"{site.name}/{cohort.name}": cohort
        for site in sorted(deriv_root.iterdir()) if site.is_dir()
        for cohort in sorted(site.iterdir()) if cohort.is_dir()
    }

def discover_data_types(dataset_path: Path) -> dict[str, Path]:
    return {p.name: p for p in sorted(dataset_path.iterdir()) if p.is_dir()}

def discover_subjects(type_path: Path) -> list[str]:
    return sorted(p.name for p in type_path.iterdir() if p.is_dir())

def file_types_for_subject(subject_dir: Path, subject_id: str) -> set[str]:
    """File presenti per un soggetto, generalizzati (subject_id -> placeholder)."""
    return {
        str(f.relative_to(subject_dir)).replace(subject_id, "{subject_id}")
        for f in subject_dir.rglob("*") if f.is_file() and f.name != ".DS_Store"
    }


### Quanti dataset ci sono

**Provenienza dei dati**

I dati grezzi vivono sotto `data/clinical_connectome/derivatives` — una cartella non versionata (gitignored), sincronizzata da EBRAIN.

Ogni **dataset** corrisponde a un centro clinico (`site`) e alla sua coorte (`cohort`), organizzati come `site/cohort` — ad esempio `UNIPD/WashU` è la coorte WashU raccolta a Padova.

La cella qui sotto scopre automaticamente questi dataset scorrendo le sottocartelle a **profondità 2** sotto `derivatives`, e ne conta **8** in totale, distribuiti su **5 centri** (UCL-UK, UKE, UKLFR, UNIPD, con UNIPD che da solo contribuisce 4 coorti).

In [3]:
datasets = discover_datasets(DERIV_ROOT)

print(f"Trovati {len(datasets)} dataset in '{DERIV_ROOT}':\n")
for name in datasets:
    print(f" - {name}")


Trovati 8 dataset in '/Users/emmatosato/Local Projects/Local PhD Projects/NEMESIS_fs/data/clinical_connectome/derivatives':

 - UCL-UK/UCLStrokeData
 - UKE/SFB936_ses01
 - UKE/WAKEUP_acute
 - UKLFR/stroke_UKLFR
 - UNIPD/NEMESIS_T0
 - UNIPD/PASPORT
 - UNIPD/PSP
 - UNIPD/WashU


### Quante tipologie di dati ci sono per dataset

**Tipologie di dato**

Per ogni dataset, la cella successiva elenca le sue **tipologie di dato disponibili** — le sottocartelle di primo livello dentro il dataset stesso (`manual_masks`, `sdc`, `features`).

Ogni tipologia corrisponde a un tipo di derivato neuroimaging diverso:

- **`manual_masks`** — le maschere di lesione segmentate manualmente
- **`sdc`** — le disconnessioni strutturali (*structural disconnectome*), calcolate a partire dalla lesione
- **`features`** — le feature funzionali già pronte (connettività, XCP-D), disponibili solo per WashU

Il risultato viene poi riassunto in una **tabella dataset × tipologia**, con `True`/`False` a indicare se quel dataset possiede quella tipologia — utile per vedere a colpo d'occhio quali coorti hanno solo la lesione e quali hanno anche SDC e/o feature funzionali.

In [4]:
presence_by_dataset = {}  # dataset -> DataFrame(subject_id x tipologia -> bool)

for name, dataset_path in datasets.items():
    types = discover_data_types(dataset_path)
    print(f"{name}: {len(types)} tipologie -> {sorted(types)}")

    subjects_by_type = {t: set(discover_subjects(p)) for t, p in types.items()}
    all_subjects = sorted(set().union(*subjects_by_type.values())) if subjects_by_type else []

    presence_by_dataset[name] = pd.DataFrame(
        {t: [s in subjects_by_type[t] for s in all_subjects] for t in sorted(types)},
        index=all_subjects,
    )


UCL-UK/UCLStrokeData: 2 tipologie -> ['manual_masks', 'sdc']
UKE/SFB936_ses01: 2 tipologie -> ['manual_masks', 'sdc']
UKE/WAKEUP_acute: 2 tipologie -> ['manual_masks', 'sdc']
UKLFR/stroke_UKLFR: 2 tipologie -> ['manual_masks', 'sdc']
UNIPD/NEMESIS_T0: 2 tipologie -> ['manual_masks', 'sdc']
UNIPD/PASPORT: 2 tipologie -> ['manual_masks', 'sdc']
UNIPD/PSP: 2 tipologie -> ['manual_masks', 'sdc']
UNIPD/WashU: 3 tipologie -> ['features', 'manual_masks', 'sdc']


In [5]:
# tabella dataset × tipologia (True/False)
all_types = sorted({t for presence in presence_by_dataset.values() for t in presence.columns})

dataset_type_availability = pd.DataFrame(
    {t: [t in presence_by_dataset[name].columns for name in datasets] for t in all_types},
    index=list(datasets),
)
display(dataset_type_availability)


,features,manual_masks,sdc
UCL-UK/UCLStrokeData,False,True,True
UKE/SFB936_ses01,False,True,True
UKE/WAKEUP_acute,False,True,True
UKLFR/stroke_UKLFR,False,True,True
UNIPD/NEMESIS_T0,False,True,True
UNIPD/PASPORT,False,True,True
UNIPD/PSP,False,True,True
UNIPD/WashU,True,True,True


### In ogni tipologia quante tipologie di dato ci sono per un soggetto

Per ogni tipologia, unione dei pattern di file su tutti i soggetti/dataset che la possiedono, poi verifica che ogni soggetto li abbia tutti. Il dettaglio si stampa solo se ci sono eccezioni:

- **assenza strutturale**: il dataset non ha mai avuto quel pattern (es. i `mapstats` solo su UKE)
  
- **assenza sporadica**: un soggetto a cui manca un pattern che gli altri del suo dataset hanno

La scansione è costosa: per ogni soggetto (~5700+) attraversa ricorsivamente la sua cartella su disco. Il risultato viene salvato in `data/clinical_connectome/file_types_cache.json` e riletto da lì nelle esecuzioni successive (vedi `OVERWRITE_FILE_TYPES_CACHE` sotto) — non va rifatta ogni volta che si riapre il notebook.</cell id="47a6c8c2">


In [6]:
import json

FILE_TYPES_CACHE_PATH = Path(project_root) / "data" / "clinical_connectome" / "file_types_cache.json"
OVERWRITE_FILE_TYPES_CACHE = False  # True per forzare la riscansione del filesystem e sovrascrivere la cache

if FILE_TYPES_CACHE_PATH.is_file() and not OVERWRITE_FILE_TYPES_CACHE:
    print(f"Carico {FILE_TYPES_CACHE_PATH} (già presente, OVERWRITE_FILE_TYPES_CACHE=False)")
    with open(FILE_TYPES_CACHE_PATH) as f:
        cached = json.load(f)
    file_types_by_type = {type_name: set(patterns) for type_name, patterns in cached["file_types_by_type"].items()}
    file_types_by_subject = {
        tuple(key.split("|")): set(patterns) for key, patterns in cached["file_types_by_subject"].items()
    }
else:
    print("Scansione del filesystem per pattern di file per soggetto (può richiedere qualche minuto)...")
    file_types_by_type = defaultdict(set)
    file_types_by_subject = {}  # (type_name, dataset, subject_id) -> set di pattern

    for name, dataset_path in datasets.items():
        for type_name, type_path in discover_data_types(dataset_path).items():
            for subject_id in discover_subjects(type_path):
                patterns = file_types_for_subject(type_path / subject_id, subject_id)
                file_types_by_type[type_name] |= patterns
                file_types_by_subject[(type_name, name, subject_id)] = patterns

    FILE_TYPES_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(FILE_TYPES_CACHE_PATH, "w") as f:
        json.dump({
            "file_types_by_type": {type_name: sorted(patterns) for type_name, patterns in file_types_by_type.items()},
            "file_types_by_subject": {"|".join(key): sorted(patterns) for key, patterns in file_types_by_subject.items()},
        }, f)
    print(f"Salvato {FILE_TYPES_CACHE_PATH} ({len(file_types_by_subject)} soggetti)")


Carico /Users/emmatosato/Local Projects/Local PhD Projects/NEMESIS_fs/data/clinical_connectome/file_types_cache.json (già presente, OVERWRITE_FILE_TYPES_CACHE=False)


In [7]:
for type_name in sorted(file_types_by_type):
    expected = file_types_by_type[type_name]
    print(f"\n=== {type_name}: {len(expected)} tipologie di file ===")

    for name, dataset_path in datasets.items():
        types = discover_data_types(dataset_path)
        if type_name not in types:
            continue
        subjects = discover_subjects(types[type_name])

        # assenze strutturali
        own_union = set().union(*(file_types_by_subject[(type_name, name, s)] for s in subjects)) if subjects else set()
        structurally_absent = expected - own_union

        # assenze sporadiche
        sporadic_missing = {
            s: (expected - file_types_by_subject[(type_name, name, s)]) - structurally_absent
            for s in subjects
        }
        sporadic_missing = {s: m for s, m in sporadic_missing.items() if m}

        if not structurally_absent and not sporadic_missing:
            print(f"✅ {name}: presenti per tutti i {len(subjects)} soggetti")
            continue

        if structurally_absent:
            print(f"ℹ️ {name}: non include {len(structurally_absent)}/{len(expected)} pattern (presenti solo in altri dataset): {sorted(structurally_absent)}")
        if sporadic_missing:
            print(f"⚠️ {name}: {len(sporadic_missing)}/{len(subjects)} soggetti con file mancanti in modo non sistematico")
            for subject_id, missing in list(sporadic_missing.items())[:10]:
                print(f"    - {subject_id}: manca {sorted(missing)}")
            if len(sporadic_missing) > 10:
                print(f"    ... +{len(sporadic_missing) - 10} altri soggetti")



=== features: 16 tipologie di file ===
✅ UNIPD/WashU: presenti per tutti i 10 soggetti

=== manual_masks: 1 tipologie di file ===
✅ UCL-UK/UCLStrokeData: presenti per tutti i 4119 soggetti
✅ UKE/SFB936_ses01: presenti per tutti i 56 soggetti
✅ UKE/WAKEUP_acute: presenti per tutti i 451 soggetti
✅ UKLFR/stroke_UKLFR: presenti per tutti i 705 soggetti
✅ UNIPD/NEMESIS_T0: presenti per tutti i 76 soggetti
✅ UNIPD/PASPORT: presenti per tutti i 83 soggetti
✅ UNIPD/PSP: presenti per tutti i 168 soggetti
✅ UNIPD/WashU: presenti per tutti i 195 soggetti

=== sdc: 34 tipologie di file ===
ℹ️ UCL-UK/UCLStrokeData: non include 33/34 pattern (presenti solo in altri dataset): ['{subject_id}_space-MNI152NLin6Asym_LF-disconnectome_atlas-aal.csv', '{subject_id}_space-MNI152NLin6Asym_LF-disconnectome_atlas-buckner_7n.csv', '{subject_id}_space-MNI152NLin6Asym_LF-disconnectome_atlas-rojkova.csv', '{subject_id}_space-MNI152NLin6Asym_LF-disconnectome_atlas-schaefer_200_7n.csv', '{subject_id}_space-MNI152NL

### Per ogni tipologia, per ogni dataset: quanti soggetti ci sono

**Cosa fa questa cella**

Per ogni dataset e tipologia, conta **quanti soggetti** sono presenti, riportando il risultato in una tabella (righe = dataset, colonne = tipologia).

Include anche un **controllo duplicati** esplicito sui nomi soggetto — ridondante in teoria (una cartella non può comparire due volte sullo stesso filesystem), ma verificato comunque come garanzia in più.

**WashU** è un caso speciale: mescola pazienti (`ST`) e controlli sani (`HC`), quindi qui viene già mostrata **corretta** — separata in due righe distinte — anziché come conteggio unico. Il dettaglio della correzione (e perché serve) è nella cella successiva.

In [8]:
rows = []
for name, dataset_path in datasets.items():
    for type_name, type_path in discover_data_types(dataset_path).items():
        subjects = discover_subjects(type_path)
        n_unique = len(set(subjects))
        rows.append({
            "dataset": name,
            "tipologia": type_name,
            "n_soggetti": len(subjects),
            "n_duplicati": len(subjects) - n_unique,
        })

# WashU: "features" su disco è solo il campione locale ridotto (10 soggetti) - la
# copertura reale (225, da features/features_archive_subjects.tsv) sostituisce le
# righe grezze, separata in ST/ HC (i controlli sani non hanno mai manual_masks/sdc).
washu_features_manifest = pd.read_csv(datasets["UNIPD/WashU"] / "features" / "features_archive_subjects.tsv", sep="\t")
washu_participants = pd.read_csv(Path(project_root) / "data" / "clinical_connectome" / "metadata_tsv" / "participants_WashU.tsv", sep="\t")
washu_disease_by_subject = washu_participants.set_index("participant_id")["disease_id"]

washu_types = discover_data_types(datasets["UNIPD/WashU"])
washu_subjects_by_type = {
    "manual_masks": set(discover_subjects(washu_types["manual_masks"])),
    "sdc": set(discover_subjects(washu_types["sdc"])),
    "features": set(washu_features_manifest["subject_id"]),
}

rows = [r for r in rows if r["dataset"] != "UNIPD/WashU"]
for group in ["ST", "HC"]:
    group_subjects = set(washu_disease_by_subject[washu_disease_by_subject == group].index)
    for type_name, subjects in washu_subjects_by_type.items():
        rows.append({
            "dataset": f"UNIPD/WashU ({group})",
            "tipologia": type_name,
            "n_soggetti": len(subjects & group_subjects),
            "n_duplicati": 0,
        })

df_counts = pd.DataFrame(rows)

# tabella per dataset (righe = dataset, colonne = tipologia, valori = n_soggetti)
df_counts_wide = df_counts.pivot(index="dataset", columns="tipologia", values="n_soggetti").astype("Int64")
row_order = [d for d in datasets if d != "UNIPD/WashU"] + ["UNIPD/WashU (ST)", "UNIPD/WashU (HC)"]

# riga "Totale": somma per tipologia su tutti i dataset (WashU già separato in ST/HC, nessun doppio conteggio)
totals = df_counts_wide.reindex(row_order).sum(numeric_only=True)
totals.name = "Totale"
display(pd.concat([df_counts_wide.reindex(row_order), totals.to_frame().T]))

dup_rows = df_counts[df_counts["n_duplicati"] > 0]
if dup_rows.empty:
    print("Nessun soggetto duplicato in nessuna tipologia.")
else:
    print(f"⚠️ Trovati {dup_rows['n_duplicati'].sum()} soggetti duplicati totali:")
    display(dup_rows[["dataset", "tipologia", "n_duplicati"]])


tipologia,features,manual_masks,sdc
UCL-UK/UCLStrokeData,<NA>,4119,4119
UKE/SFB936_ses01,<NA>,56,56
UKE/WAKEUP_acute,<NA>,451,451
UKLFR/stroke_UKLFR,<NA>,705,705
UNIPD/NEMESIS_T0,<NA>,76,76
UNIPD/PASPORT,<NA>,83,83
UNIPD/PSP,<NA>,168,168
UNIPD/WashU (ST),169,195,195
UNIPD/WashU (HC),56,0,0
Totale,225,5853,5853


Nessun soggetto duplicato in nessuna tipologia.


#### Correzione soggetti per WashU

`features` su disco per WashU è solo il campione locale ridotto (10 soggetti) — la copertura reale (225) viene dal manifest verificato dell'archivio compresso (`features/features_archive_subjects.tsv`, vedi `README_ARCHIVE.md` nella stessa cartella). 

WashU inoltre mescola pazienti (ST) e controlli sani (HC, che per costruzione non hanno mai `manual_masks`/`sdc` — nessuna lesione da segnare): li separiamo in due dataset così i confronti sotto restano sensati.

In [9]:
washu_features_manifest = pd.read_csv(datasets["UNIPD/WashU"] / "features" / "features_archive_subjects.tsv", sep="\t")
washu_real_features_subjects = set(washu_features_manifest["subject_id"])

washu_participants = pd.read_csv(Path(project_root) / "data" / "clinical_connectome" / "metadata_tsv" / "participants_WashU.tsv", sep="\t")
washu_disease_by_subject = washu_participants.set_index("participant_id")["disease_id"]

washu_types = discover_data_types(datasets["UNIPD/WashU"])
manual_masks_subjects = set(discover_subjects(washu_types["manual_masks"]))
sdc_subjects = set(discover_subjects(washu_types["sdc"]))

all_washu_subjects = sorted(set(presence_by_dataset["UNIPD/WashU"].index) | washu_real_features_subjects)
washu_corrected = pd.DataFrame({
    "manual_masks": [s in manual_masks_subjects for s in all_washu_subjects],
    "sdc": [s in sdc_subjects for s in all_washu_subjects],
    "features": [s in washu_real_features_subjects for s in all_washu_subjects],
}, index=all_washu_subjects)

washu_groups = washu_disease_by_subject.reindex(washu_corrected.index)
if washu_groups.isna().any():
    raise ValueError(f"disease_id non trovato per: {washu_groups[washu_groups.isna()].index.tolist()}")

del presence_by_dataset["UNIPD/WashU"]
presence_by_dataset["UNIPD/WashU (ST)"] = washu_corrected[washu_groups == "ST"]
presence_by_dataset["UNIPD/WashU (HC)"] = washu_corrected[washu_groups == "HC"]

for name in ["UNIPD/WashU (ST)", "UNIPD/WashU (HC)"]:
    print(f"{name}: {len(presence_by_dataset[name])} soggetti")
    print(presence_by_dataset[name].sum())
    print()


UNIPD/WashU (ST): 195 soggetti
manual_masks    195
sdc             195
features        169
dtype: int64

UNIPD/WashU (HC): 56 soggetti
manual_masks     0
sdc              0
features        56
dtype: int64



### Confronto a coppie tra due tipologie

Scegli due tipologie (`MOD_A`, `MOD_B`): per ogni dataset che le ha entrambe, elenca i soggetti presenti in una ma non nell'altra, e viceversa — mostrati solo i dataset con almeno un'eccezione.

In [10]:
MOD_A = "manual_masks"
MOD_B = "sdc"

for name, presence in presence_by_dataset.items():
    if MOD_A not in presence.columns or MOD_B not in presence.columns:
        continue

    only_a = presence.index[presence[MOD_A] & ~presence[MOD_B]]
    only_b = presence.index[~presence[MOD_A] & presence[MOD_B]]
    if only_a.empty and only_b.empty:
        continue

    print(f"\n=== {name} ===")
    if not only_a.empty:
        print(f"In '{MOD_A}' ma non in '{MOD_B}' ({len(only_a)}): {list(only_a)}")
    if not only_b.empty:
        print(f"In '{MOD_B}' ma non in '{MOD_A}' ({len(only_b)}): {list(only_b)}")
